In [80]:
import pandas as pd
import sqlite3

In [81]:
conn = sqlite3.connect('../data/checking-logs.sqlite')

test = pd.read_sql("PRAGMA table_info(test);", conn)
test

,cid,name,type,notnull,dflt_value,pk
0,0,uid,TEXT,0,None,0
1,1,labname,TEXT,0,None,0
2,2,first_commit_ts,TIMESTAMP,0,None,0
3,3,first_view_ts,TIMESTAMP,0,None,0


In [82]:
pd.read_sql("select * from test limit 10;", conn)

,uid,labname,first_commit_ts,first_view_ts
0,user_1,laba04,2020-04-26 17:06:18.462708,2020-04-26 21:53:59.624136
1,user_1,laba04s,2020-04-26 17:12:11.843671,2020-04-26 21:53:59.624136
2,user_1,laba05,2020-05-02 19:15:18.540185,2020-04-26 21:53:59.624136
3,user_1,laba06,2020-05-17 16:26:35.268534,2020-04-26 21:53:59.624136
4,user_1,laba06s,2020-05-20 12:23:37.289724,2020-04-26 21:53:59.624136
5,user_1,project1,2020-05-14 20:56:08.898880,2020-04-26 21:53:59.624136
6,user_10,laba04,2020-04-25 08:24:52.696624,2020-04-18 12:19:50.182714
7,user_10,laba04s,2020-04-25 08:37:54.604222,2020-04-18 12:19:50.182714
8,user_10,laba05,2020-05-01 19:27:26.063245,2020-04-18 12:19:50.182714
9,user_10,laba06,2020-05-19 11:39:28.885637,2020-04-18 12:19:50.182714


In [83]:
query = """
select t.uid, min((julianday(datetime(d.deadlines, 'unixepoch')) - julianday(first_commit_ts)) * 24) as diff_hours
from test t
join deadlines d on t.labname = d.labs
where t.labname != 'project1'
group by t.uid
"""

df_min = pd.read_sql(query, conn)
df_min

,uid,diff_hours
0,user_1,6.796433
1,user_10,39.367888
2,user_14,84.448466
3,user_17,34.643043
4,user_18,3.933907
5,user_19,32.729282
6,user_21,33.905274
7,user_25,2.867236
8,user_28,8.103915
9,user_3,60.511392


In [84]:
query = """
select t.uid, max((julianday(datetime(d.deadlines, 'unixepoch')) - julianday(first_commit_ts)) * 24) as diff_hours
from test t
join deadlines d on t.labname = d.labs
where t.labname != 'project1'
group by t.uid
"""

df_max = pd.read_sql(query, conn)
df_max

,uid,diff_hours
0,user_1,175.556592
1,user_10,132.341699
2,user_14,200.766302
3,user_17,81.591404
4,user_18,10.973376
5,user_19,148.916029
6,user_21,126.199587
7,user_25,150.869726
8,user_28,174.852984
9,user_3,182.055144


In [85]:
query = """
select avg((julianday(datetime(d.deadlines, 'unixepoch')) - julianday(first_commit_ts)) * 24) as diff_hours
from test t
join deadlines d on t.labname = d.labs
where t.labname != 'project1'
"""

df_avg = pd.read_sql(query, conn)
df_avg

,diff_hours
0,89.687686


In [86]:
query = """
select t.uid, 
avg((julianday(datetime(d.deadlines, 'unixepoch')) - julianday(first_commit_ts)) * 24) as avg_diff,
count(p.datetime) as pageviews
from test t
join deadlines d on t.labname = d.labs
left join pageviews p on t.uid = p.uid
where t.labname != 'project1'
group by t.uid
"""

views = pd.read_sql(query, conn)
views

,uid,avg_diff,pageviews
0,user_1,65.119644,140
1,user_10,75.242310,445
2,user_14,159.568696,429
3,user_17,62.207514,235
4,user_18,6.367907,9
5,user_19,99.440298,64
6,user_21,96.111042,40
7,user_25,93.474751,895
8,user_28,86.793652,745
9,user_3,105.738041,1585


In [87]:
correl = views[['pageviews', 'avg_diff']].corr()
correl.loc['pageviews', 'avg_diff']

np.float64(0.1850419938265189)

In [88]:
conn.close()